# 第 13 讲：仿真模型与蒙特卡洛

> 用蒙特卡洛处理不确定性，输出结果分布、置信区间和决策建议。

## 课件导学

**任务情境**：当需求、成本或风险不确定时，仿真比单点计算更能说明决策风险。

**关键概念**

- 随机变量
- 蒙特卡洛
- 置信区间
- 库存决策
- 风险收益

**本讲产物**

- monte_carlo_inventory.csv
- monte_carlo_inventory.png

## 资料链接与阅读抓手

下面这些链接优先选官方文档或稳定资料。课前先看标题和示例，课堂中遇到参数或概念再回查。

- [NumPy random 官方指南](https://numpy.org/doc/stable/reference/random/index.html)：学习使用 Generator 生成随机样本。
- [NumPy Generator 文档](https://numpy.org/doc/2.3/reference/random/generator.html)：查看不同分布和 size 参数。
- [SciPy statistics](https://docs.scipy.org/doc/scipy/reference/stats.html)：后续可扩展到更多概率分布。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-13"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

## 可运行示例与讲解路线

In [ ]:
def simulate_inventory(order_qty, n_sim=5000):
    demand = rng.normal(100, 18, n_sim).clip(0, None)
    price, cost, salvage, shortage_penalty = 12, 7, 3, 2
    sales = np.minimum(order_qty, demand)
    leftover = np.maximum(order_qty - demand, 0)
    shortage = np.maximum(demand - order_qty, 0)
    profit = price * sales + salvage * leftover - cost * order_qty - shortage_penalty * shortage
    return profit

quantities = np.arange(60, 151, 5)
rows = []
for q in quantities:
    profit = simulate_inventory(q)
    rows.append({
        "order_qty": q,
        "mean_profit": profit.mean(),
        "p05": np.percentile(profit, 5),
        "p95": np.percentile(profit, 95),
    })
result = pd.DataFrame(rows)
best = result.loc[result["mean_profit"].idxmax()]
result.to_csv(OUTPUT_ROOT / "monte_carlo_inventory.csv", index=False)
best

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(result["order_qty"], result["mean_profit"], marker="o", label="mean profit")
ax.fill_between(result["order_qty"], result["p05"], result["p95"], alpha=0.2, label="90% interval")
ax.axvline(best["order_qty"], color="red", linestyle="--", label="best")
ax.set_xlabel("Order quantity")
ax.set_ylabel("Profit")
ax.set_title("Monte Carlo inventory decision")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "monte_carlo_inventory.png", dpi=160)
plt.show()

## 实战环节

**课堂任务**

- 改变需求方差，观察最优订货量变化。
- 比较均值收益和 5% 分位收益。
- 写出风险厌恶型决策建议。

**课后挑战**：把目标从最大均值收益改成最大 5% 分位收益。

**验收清单**

- 是否固定随机种子
- 模拟次数是否足够
- 是否报告区间而不是只报告均值

In [ ]:
lesson_resources = [{'title': 'NumPy random 官方指南', 'url': 'https://numpy.org/doc/stable/reference/random/index.html', 'reading_note': '学习使用 Generator 生成随机样本。'}, {'title': 'NumPy Generator 文档', 'url': 'https://numpy.org/doc/2.3/reference/random/generator.html', 'reading_note': '查看不同分布和 size 参数。'}, {'title': 'SciPy statistics', 'url': 'https://docs.scipy.org/doc/scipy/reference/stats.html', 'reading_note': '后续可扩展到更多概率分布。'}]
lesson_deliverables = ['monte_carlo_inventory.csv', 'monte_carlo_inventory.png']
lesson_checklist = ['是否固定随机种子', '模拟次数是否足够', '是否报告区间而不是只报告均值']

pd.DataFrame(lesson_resources).to_csv(OUTPUT_ROOT / "lesson_resources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"deliverable": lesson_deliverables}).to_csv(OUTPUT_ROOT / "lesson_deliverables.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"check_item": lesson_checklist}).to_csv(OUTPUT_ROOT / "lesson_checklist.csv", index=False, encoding="utf-8-sig")
print("Saved courseware resource map, deliverables, and checklist.")